### Import Required Libraries
Load essential Python libraries:
- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computations
- **glob**: File pattern matching for CSV file discovery
- **os**: Operating system utilities for path handling

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

### Define File Paths
Configure input and output paths:
- **csv_folder**: Directory containing the HUPA-UC Diabetes Dataset CSV files
- **output_file**: Destination path for the combined and processed data file

In [ ]:
csv_folder = r'C:\Users\ruchi\OneDrive\Documents\Python Hackathon May 2026\HUPA-UC Diabetes Dataset-20250820T010637Z-1-001\HUPA-UC Diabetes Dataset'
output_file = r"C:\Temporary\Combined_data.csv"

### Read and Combine CSV Files
Aggregate multiple HUPA dataset files into a single DataFrame:
1. Locate all CSV files matching the HUPA pattern in the source folder
2. Read each CSV file using semicolon as delimiter
3. Add a "source_file" column to track which file each record originated from
4. Concatenate all DataFrames into one combined dataset
5. Save the combined data to the output file

In [ ]:
files = glob.glob(os.path.join(csv_folder, "HUPA*.csv"))

combined_list = []
for file in files:
    temp = pd.read_csv(file,sep=";")
    temp["source_file"] = os.path.basename(file)   # optional but useful
    combined_list.append(temp)

combined = pd.concat(combined_list, ignore_index=True)
combined.to_csv(output_file, index=False)

print("Combined CSV created:", output_file)

In [ ]:
# Check column names
print(combined.columns.tolist())

### Mapping to new column and Rename Columns
Transform column names to more descriptive and standardized format:
- Map raw column names to human-readable equivalents with units
- Examples: `glucose` → `Blood_Glucose_mg_dl`, `calories` → `Calories_burned`
- Ensure consistent naming convention across the dataset
- Save updated dataset and verify new column names  

In [ ]:

# combined.columns = combined.columns.str.strip().str.title()

column_rename_map = {
   "time":                  "TimeStamp",
    "glucose":        "Blood_Glucose_mg_dl",
    "calories":            "Calories_burned",
    "heart_rate":             "Heart_Rate_bpm",
    "steps":                 "Step_count",
    "basal_rate": "Basal_Insulin_Rate_Unit_hr",
    "bolus_volume_delivered":   "Bolus_Insulin_Dose_Units",
    "carb_input":  "Carbohydrate_Intake_Grams",
    "source_file":                "Patient_ID"
}

combined = combined.rename(columns=column_rename_map)
combined.to_csv(output_file, index=False)
print("Saved successfully!")
print(combined.columns.tolist())


### Clean Patient ID
Normalize the Patient_ID column:
- Remove the ".csv" file extension suffix from patient identifiers
- Display a sample of cleaned IDs to verify the transformation
- Save the cleaned dataset

In [ ]:
combined["Patient_ID"] = combined["Patient_ID"].str.replace(".csv", "", regex=False)
combined.to_csv(output_file, index=False)
combined["Patient_ID"].head()

### Drop Unnecessary Columns
Remove demographic and sleep-related columns not required for analysis:
- **Demographic fields**: age, gender, race
- **Sleep metrics**: average sleep duration, sleep quality rating, sleep disturbance percentage
- **Redundant ID**: duplicate patient_id column
- Retain only health measurement and temporal data

In [ ]:
cols_to_drop = ['age', 'gender', 'race', 'average sleep duration (hrs)', 'sleep quality (1-10)', '% with sleep disturbances', 'patient_id']

combined = combined.drop(columns=[col for col in cols_to_drop if col in combined.columns])
combined.to_csv(output_file, index=False)
print(combined.columns.tolist())

### Verify Final Columns
Display the complete list of columns remaining after all data cleaning and transformation steps.


In [ ]:

print(combined.columns.tolist())


### Parse Timestamp and Extract Temporal Features
Convert timestamp data into structured temporal components for time-based analysis:
- Parse the TimeStamp column to datetime format for proper time handling
- Extract **Date**: calendar date component
- Extract **Time**: clock time component
- Extract **Hour**: hour of day (0-23) for time-of-day pattern analysis
- These features enable temporal analysis (e.g., blood glucose patterns by time of day)

In [ ]:
combined["TimeStamp"] = pd.to_datetime(combined["TimeStamp"])
combined["Date"]     = combined["TimeStamp"].dt.date
combined["Time"]     = combined["TimeStamp"].dt.time
combined["Hour"]     = combined["TimeStamp"].dt.hour

combined

### Display Final Columns
Review the complete and final set of columns including newly extracted temporal features (Date, Time, Hour) ready for analysis.

In [ ]:

print(combined.columns.tolist())
